# Moderate v3: landmark mouth-crop preparation

This notebook prepares the moderate timing candidate for the same controlled lip-reading experiments. It does not train a model.

Pipeline: raw word clips -> MediaPipe landmark mouth crops -> speaker split -> Colab training archive.

The source archive contains only split-referenced clips and the existing semester-4 utilities.

## 1. Install dependencies

MediaPipe and OpenCV are needed only for this preprocessing stage.

In [ ]:
%pip -q install mediapipe opencv-python-headless

## 2. Mount Drive and extract the source archive

Upload `colab_lipreading_moderate_v3_prep.zip` to `MyDrive/lipreading_sem4/` before running this cell.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import zipfile
from google.colab import drive

drive.mount('/content/drive')

archive = Path('/content/drive/MyDrive/lipreading_sem4/colab_lipreading_moderate_v3_prep.zip')
extract_dir = Path('/content/lipreading_moderate_v3')
if not archive.is_file():
    raise FileNotFoundError(f'Archive not found: {archive}')

if not extract_dir.exists():
    extract_dir.mkdir(parents=True)
    base = extract_dir.resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (extract_dir / member.filename).resolve()
            if base not in target.parents and target != base:
                raise RuntimeError(f'Unsafe archive member: {member.filename}')
        z.extractall(extract_dir)

dataset_root = extract_dir / 'ru_dataset'
split_dir = dataset_root / '03_splits/ml_splits_moderate_v3/speaker_top10'
tool_dir = extract_dir / 'tools'
print('dataset_root:', dataset_root)
print('source train/val/test:', [p.exists() for p in [split_dir / 'train.csv', split_dir / 'val.csv', split_dir / 'test.csv']])

## 3. Create landmark mouth crops

The same existing `create_landmark_mouth_crops.py` is run on train, validation and test. A crop is accepted when at least 70% of its frames have detected landmarks.

In [ ]:
tool = tool_dir / 'create_landmark_mouth_crops.py'
out_dir = dataset_root / '02_model_inputs/mouth_crops_landmark_moderate_v3'
manifest = dataset_root / '04_quality_reports/landmark_moderate_v3_labels.csv'
failed = dataset_root / '04_quality_reports/landmark_moderate_v3_failed.csv'
contact = dataset_root / '04_quality_reports/landmark_moderate_v3_contact_sheet.jpg'

command = [
    'python', str(tool),
    '--dataset-root', str(dataset_root),
    '--split-csv', str(split_dir / 'train.csv'),
    '--split-csv', str(split_dir / 'val.csv'),
    '--split-csv', str(split_dir / 'test.csv'),
    '--out-dir', str(out_dir),
    '--manifest', str(manifest),
    '--failed-manifest', str(failed),
    '--contact-sheet', str(contact),
    '--size', '96',
    '--min-detection-ratio', '0.70',
]
subprocess.run(command, check=True)
print('landmark manifest:', manifest)

## 4. Audit crop coverage

Do not delete failed source clips automatically. First inspect the failed manifest and the contact sheet.

In [ ]:
import pandas as pd

landmark_df = pd.read_csv(manifest)
failed_df = pd.read_csv(failed) if failed.exists() else pd.DataFrame()
print('success:', len(landmark_df))
print('failed:', len(failed_df))
print('status:', landmark_df['landmark_status'].value_counts().to_dict())
print(landmark_df['landmark_detection_ratio'].astype(float).describe())
assert set(landmark_df['word']) == set(pd.read_csv(split_dir / 'vocab.txt', header=None)[0])
assert len(landmark_df) >= 0.90 * 818, 'Less than 90% of moderate clips passed landmark preprocessing'

## 5. Build a new speaker split and package mouth crops

The split remains speaker-independent: `spk16,spk17` train, `spk18` validation, `spk19` test.

In [ ]:
split_out = dataset_root / '03_splits/ml_splits_landmark_moderate_v3'
subprocess.run([
    'python', str(tool_dir / 'create_ml_splits.py'),
    '--labels', str(manifest),
    '--out-dir', str(split_out),
    '--top-words', '10',
    '--seed', '42',
    '--train-speakers', 'spk16,spk17',
    '--val-speakers', 'spk18',
    '--test-speakers', 'spk19',
    '--only-speaker-top',
], check=True)

mouth_archive = Path('/content/colab_lipreading_moderate_v3_landmark.zip')
subprocess.run([
    'python', str(tool_dir / 'package_ml_dataset.py'),
    '--dataset-root', str(dataset_root),
    '--split-dir', str(split_out / 'speaker_top10'),
    '--output', str(mouth_archive),
    '--archive-root', 'ru_dataset',
], check=True)

drive_output = Path('/content/drive/MyDrive/lipreading_sem4') / mouth_archive.name
shutil.copy2(mouth_archive, drive_output)
print('training archive:', drive_output)

## Next

Use `colab_lipreading_moderate_v3_landmark.zip` with the existing controlled MS-TCN/BiGRU training notebook. Keep the same seed set, epochs, augmentation and metrics as the padded experiment. The result is a boundary-preprocessing comparison, not a new test-selection opportunity.